# Embeddings y preparación de texto (Capítulo 2)

Este notebook reproduce el flujo central del capítulo 2 de *Build a Large Language Model (From Scratch)* y lo adapta para que corra de punta a punta sin errores.

## 1) Por qué este paso importa para LLMs y sistemas agénticos

Un LLM no consume texto crudo: primero hay que convertirlo en IDs y luego en vectores. En agentes, esta capa es crítica porque una mala segmentación del texto degrada recuperación, planificación y uso de herramientas.

In [1]:
from importlib.metadata import version
import re
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

print('torch:', version('torch'))
print('tiktoken:', version('tiktoken'))

torch: 2.8.0
tiktoken: 0.12.0


In [2]:
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

print('Total caracteres:', len(raw_text))
print('Primeros 180 chars:\n', raw_text[:180])

Total caracteres: 20479
Primeros 180 chars:
 I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his


In [3]:
# Tokenizacion estilo capitulo: separar puntuacion y espacios
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_tokens = sorted(set(preprocessed))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])
vocab = {token: idx for idx, token in enumerate(all_tokens)}

print('Tokens totales:', len(preprocessed))
print('Vocabulario:', len(vocab))
print('Primeros 20 tokens:', preprocessed[:20])

Tokens totales: 4690
Vocabulario: 1132
Primeros 20 tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']


## 2) Tokenizador robusto y OOV

La versión simple falla con palabras fuera de vocabulario. En producción (y en agentes), siempre habrá OOV (out-of-vocabulary), así que mapear a `<|unk|>` evita romper el pipeline.

In [4]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        pre = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        pre = [item.strip() for item in pre if item.strip()]
        pre = [item if item in self.str_to_int else '<|unk|>' for item in pre]
        return [self.str_to_int[s] for s in pre]

    def decode(self, ids):
        text = ' '.join([self.int_to_str[i] for i in ids])
        text = re.sub(r"\s+([,.?\"()!'])", r'\1', text)
        return text

tokenizer_v2 = SimpleTokenizerV2(vocab)
sample_text = 'Hello, do you like tea? In the sunlit terraces of the palace.'
encoded = tokenizer_v2.encode(sample_text)
decoded = tokenizer_v2.decode(encoded)

print('Encoded:', encoded[:20])
print('Decoded:', decoded)

Encoded: [1131, 5, 355, 1126, 628, 975, 10, 55, 988, 956, 984, 722, 988, 1131, 7]
Decoded: <|unk|>, do you like tea? In the sunlit terraces of the <|unk|>.


## 3) BPE y ventana deslizante

El libro usa `tiktoken` (BPE) porque reduce OOV al descomponer palabras en subunidades. Después, la ventana deslizante crea pares `(input, target)` para entrenamiento autoregresivo (predecir el siguiente token).

In [5]:
tokenizer = tiktoken.get_encoding('gpt2')
enc_text = tokenizer.encode(raw_text)
print('Tokens GPT-2:', len(enc_text))
print('Primeros 16 token IDs:', enc_text[:16])

Tokens GPT-2: 5145
Primeros 16 token IDs: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257]


In [6]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=False, drop_last=True):
    tok = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tok, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last), dataset

dataloader, dataset = create_dataloader_v1(raw_text, batch_size=2, max_length=8, stride=4, shuffle=False)
x, y = next(iter(dataloader))
print('Batch input shape:', x.shape)
print('Batch target shape:', y.shape)
print('Primer input:', x[0].tolist())
print('Primer target:', y[0].tolist())

Batch input shape: torch.Size([2, 8])
Batch target shape: torch.Size([2, 8])
Primer input: [40, 367, 2885, 1464, 1807, 3619, 402, 271]
Primer target: [367, 2885, 1464, 1807, 3619, 402, 271, 10899]


## 4) Experimento: cambiar `max_length` y `stride`

Más `max_length` produce menos ventanas porque cada muestra consume más tokens. Un `stride` pequeño genera traslape (overlap), aumentando muestras y continuidad de contexto entre ejemplos consecutivos.

In [7]:
def sample_count(txt, max_length, stride):
    tok = tiktoken.get_encoding('gpt2')
    n = len(tok.encode(txt))
    return len(range(0, n - max_length, stride))

configs = [
    (4, 4),
    (4, 2),
    (8, 4),
    (16, 8),
]

print('max_length | stride | num_samples')
for m, s in configs:
    print(f'{m:10d} | {s:6d} | {sample_count(raw_text, m, s):11d}')

max_length | stride | num_samples
         4 |      4 |        1286
         4 |      2 |        2571
         8 |      4 |        1285
        16 |      8 |         642


Con este texto, bajar `stride` de 4 a 2 (con el mismo `max_length=4`) incrementa el número de muestras porque movemos la ventana menos posiciones cada vez.

El overlap es útil porque: (1) reutiliza transiciones locales importantes, (2) suaviza bordes entre chunks, y (3) mejora cobertura de contextos para entrenamiento de next-token prediction.

## 5) Embeddings: significado y relación con NN

Los embeddings codifican significado porque durante entrenamiento las palabras/subtokens que aparecen en contextos parecidos reciben gradientes parecidos y terminan con vectores cercanos en el espacio latente.

Desde redes neuronales, una capa `nn.Embedding` es una matriz de parámetros entrenables (lookup table). Cada token ID selecciona una fila de esa matriz. Esa fila se ajusta por backpropagation igual que cualquier peso de una NN.

In [8]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(64, output_dim)

dl, _ = create_dataloader_v1(raw_text, batch_size=2, max_length=8, stride=8, shuffle=False)
inputs, targets = next(iter(dl))

token_embeddings = token_embedding_layer(inputs)
pos_ids = torch.arange(inputs.shape[1])
pos_embeddings = pos_embedding_layer(pos_ids)
input_embeddings = token_embeddings + pos_embeddings

print('Token IDs shape:', inputs.shape)
print('Token embeddings shape:', token_embeddings.shape)
print('Pos embeddings shape:', pos_embeddings.shape)
print('Input embeddings shape:', input_embeddings.shape)

Token IDs shape: torch.Size([2, 8])
Token embeddings shape: torch.Size([2, 8, 256])
Pos embeddings shape: torch.Size([8, 256])
Input embeddings shape: torch.Size([2, 8, 256])
